In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Création de la session Spark en mode local
spark = (
    SparkSession.builder
    .appName("DemoSparkProducts")
    .master("local[*]")
    .config(
        "spark.jars.packages",
        "org.mongodb.spark:mongo-spark-connector_2.13:11.1.0"
    )
    .config(
        "spark.mongodb.read.connection.uri",
        "mongodb://localhost:27017/commerce.raw_products"
    )
    .getOrCreate()
)

# Lecture de la collection MongoDB raw_products
df_products = (
    spark.read.format("mongodb")
    .option("database", "commerce")
    .option("collection", "raw_products")
    .load()
)

print("Version Spark :", spark.version)
print("Spark UI :", spark.sparkContext.uiWebUrl)

Py4JJavaError: An error occurred while calling None.org.apache.spark.api.java.JavaSparkContext.
: java.lang.RuntimeException: java.io.FileNotFoundException: java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset. -see https://cwiki.apache.org/confluence/display/HADOOP2/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:789)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:298)
	at org.apache.hadoop.fs.FileUtil.chmod(FileUtil.java:1305)
	at org.apache.hadoop.fs.FileUtil.chmod(FileUtil.java:1291)
	at org.apache.spark.util.Utils$.fetchFile(Utils.scala:417)
	at org.apache.spark.SparkContext.addFile(SparkContext.scala:1864)
	at org.apache.spark.SparkContext.$anonfun$new$16(SparkContext.scala:545)
	at org.apache.spark.SparkContext.$anonfun$new$16$adapted(SparkContext.scala:545)
	at scala.collection.immutable.List.foreach(List.scala:334)
	at org.apache.spark.SparkContext.<init>(SparkContext.scala:545)
	at org.apache.spar

In [2]:
# Afficher la structure du DataFrame
df_products.printSchema()

# Afficher les 5 premiers produits
df_products.show(5, truncate=False)

# Compter le nombre total de produits
print("Nombre de produits :", df_products.count())

root
 |-- _id: string (nullable = true)
 |-- availabilityStatus: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- category: string (nullable = true)
 |-- description: string (nullable = true)
 |-- dimensions: struct (nullable = true)
 |    |-- width: double (nullable = true)
 |    |-- height: double (nullable = true)
 |    |-- depth: double (nullable = true)
 |-- discountPercentage: double (nullable = true)
 |-- id: integer (nullable = true)
 |-- images: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- meta: struct (nullable = true)
 |    |-- createdAt: string (nullable = true)
 |    |-- updatedAt: string (nullable = true)
 |    |-- barcode: string (nullable = true)
 |    |-- qrCode: string (nullable = true)
 |-- minimumOrderQuantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- rating: double (nullable = true)
 |-- returnPolicy: string (nullable = true)
 |-- reviews: array (nullable = true)
 |    |-- element: struct 

+------------------------+------------------+--------------+--------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+---------------------+------------------+---+--------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------+--------------------+-----+------+--------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------+---------------+-----+---------------------+-----------------------------------------------------------------------------

In [3]:
# Sélectionner les colonnes utiles et filtrer les produits
df_expensive = (
    df_products
    .select("title", "category", "price", "rating", "stock")
    .filter(F.col("price") > 100)
)

# Afficher les 10 premiers résultats
df_expensive.show(10, truncate=False)

+-----------------------------------------+----------+-------+------+-----+
|title                                    |category  |price  |rating|stock|
+-----------------------------------------+----------+-------+------+-----+
|Chanel Coco Noir Eau De                  |fragrances|129.99 |4.26  |58   |
|Annibale Colombo Bed                     |furniture |1899.99|4.77  |88   |
|Annibale Colombo Sofa                    |furniture |2499.99|3.92  |60   |
|Bedside Table African Cherry             |furniture |299.99 |2.87  |64   |
|Knoll Saarinen Executive Conference Chair|furniture |499.99 |4.88  |26   |
|Wooden Bathroom Sink With Mirror         |furniture |799.99 |3.59  |7    |
|Apple MacBook Pro 14 Inch Space Grey     |laptops   |1999.99|3.65  |24   |
|Asus Zenbook Pro Dual Screen Laptop      |laptops   |1799.99|3.95  |45   |
|Huawei Matebook X Pro                    |laptops   |1399.99|4.98  |75   |
|Lenovo Yoga 920                          |laptops   |1099.99|2.86  |40   |
+-----------

In [4]:
# Calculer des statistiques pour chaque catégorie
df_category_stats = (
    df_products
    .groupBy("category")
    .agg(
        F.count("*").alias("nombre_produits"),
        F.round(F.avg("price"), 2).alias("prix_moyen"),
        F.round(F.avg("rating"), 2).alias("note_moyenne")
    )
    .orderBy(F.desc("nombre_produits"))
)

# Afficher les résultats
df_category_stats.show(20, truncate=False)

+-------------------+---------------+----------+------------+
|category           |nombre_produits|prix_moyen|note_moyenne|
+-------------------+---------------+----------+------------+
|kitchen-accessories|30             |17.69     |3.82        |
|groceries          |27             |5.47      |3.91        |
|sports-accessories |17             |24.11     |4.01        |
|smartphones        |16             |424.99    |3.7         |
|mobile-accessories |14             |146.99    |3.85        |
|mens-watches       |6              |8098.32   |3.66        |
|motorcycle         |5              |7699.99   |3.67        |
|womens-watches     |5              |6585.99   |3.48        |
|mens-shoes         |5              |109.99    |4.6         |
|fragrances         |5              |83.99     |3.83        |
|beauty             |5              |13.39     |3.75        |
|sunglasses         |5              |26.59     |3.73        |
|womens-bags        |5              |175.99    |3.36        |
|womens-

In [5]:
# Construire un traitement sans déclencher immédiatement le calcul
df_demo = (
    df_products
    .filter(F.col("stock") > 0)
    .select("title", "category", "price", "stock")
)

# Afficher le plan d'exécution
df_demo.explain()

# Observer le nombre de partitions
print("Nombre de partitions :", df_demo.rdd.getNumPartitions())

# Déclencher le calcul avec une action
print("Produits en stock :", df_demo.count())

== Physical Plan ==
*(1) Project [title#20, category#3, price#11, stock#17]
+- BatchScan MongoTable()[category#3, price#11, stock#17, title#20] MongoScan{namespaceDescription=commerce.raw_products} RuntimeFilters: []


Nombre de partitions : 1
Produits en stock : 190
